# Multi-Agent System for Root Cause Analysis

This notebook contains the main agent wrapper and MLflow integration for the multi-agent system.


In [0]:
%pip install -r ../requirements.txt

In [0]:
dbutils.library.restartPython()

In [0]:
import sys

from multiAgentSystem.deps import get_deps
from multiAgentSystem.config import LLM_ENDPOINT_NAME, MAX_OUTER_ITERATIONS, MAX_ANALYZE_PARSE_LOOPS

print(f"LLM Endpoint: {LLM_ENDPOINT_NAME}")
print(f"Max Iterations: {MAX_OUTER_ITERATIONS}")
print(f"Inner Interations: {MAX_ANALYZE_PARSE_LOOPS}")

In [0]:
"""Utilities for constructing and running the RCA agent."""
from __future__ import annotations

from typing import Dict, Any, Generator, Optional

from multiAgentSystem.state import AgentState

from multiAgentSystem.graph import build_graph
from multiAgentSystem.deps import get_deps, reset_deps
from multiAgentSystem.config import (
    MLFLOW_ENABLED,
    MAX_OUTER_ITERATIONS,
    MAX_ANALYZE_PARSE_LOOPS,
    CONFIDENCE_THRESHOLD,
    GRAPH_RECURSION_LIMIT,
    configure_agent_llms,
    show_llm_configuration,
)


class RCAAgent:
    """Thin wrapper exposing ``predict`` and ``predict_stream`` helpers.
    
    Args:
        graph: Optional pre-built graph. If None, builds a new one.
        llm_config: Optional dict mapping agent names to LLM endpoints.
                   Example: {"reasoning": "databricks-claude-sonnet-4-5",
                            "supervisor": "databricks-claude-opus"}
    """

    def __init__(self, graph=None, llm_config: Optional[Dict[str, str]] = None):
        # Configure LLMs if provided
        if llm_config:
            configure_agent_llms(llm_config)
            # Reset dependencies to pick up new config
            reset_deps()
        
        self.graph = graph or build_graph()

    def _init_state(self, request: Dict[str, Any]) -> AgentState:
        """Initialise the agent state from the inbound request payload."""
        user_context = request.get("user_context")
        if not user_context:
            msgs = request.get("input") or []
            if isinstance(msgs, list):
                user_parts = [
                    m.get("content", "")
                    for m in msgs
                    if isinstance(m, dict) and m.get("role") == "user"
                ]
                user_context = "\n\n".join([p for p in user_parts if p]) or ""

        logs_path = request.get("logs_path", "") or request.get("path", "") or ""

        return AgentState(
            user_context=user_context or "",
            logs_path=logs_path,
            iteration=0,
            hypotheses=[],
            keywords=[],
            # NEW: Optimized evidence storage
            evidence_map={},
            evidence_summary="",
            last_logs_chunk="",
            analyzer_satisfied=False,
            last_generated_keywords=[],
            draft={"problem": "", "rca": "", "mitigation": ""},
            confidence=0.0,
            critic_approved=False,
            critique="",
            last_status="",
            next_action="",
            supervisor_rationale="",
            analyze_parse_loops=0,
            pdf_report_path=None,
        )

    def predict(self, request: Dict[str, Any]) -> Dict[str, Any]:
        """Run the multi-agent system and return final results."""
        final_state: AgentState = self.graph.invoke(
            self._init_state(request),
            config={"recursion_limit": GRAPH_RECURSION_LIMIT}
        )
        return {
            "output": {
                "problem": final_state.get("draft", {}).get("problem", ""),
                "rca": final_state.get("draft", {}).get("rca", ""),
                "mitigation": final_state.get("draft", {}).get("mitigation", ""),
                "confidence": float(final_state.get("confidence", 0.0)),
                "iterations": int(final_state.get("iteration", 0)),
                "keywords": final_state.get("keywords", []),
                # NEW: Include evidence_map and evidence_summary in output
                "evidence_map": final_state.get("evidence_map", {}),
                "evidence_summary": final_state.get("evidence_summary", ""),
                "critic_approved": bool(final_state.get("critic_approved", False)),
                "critique": final_state.get("critique", ""),
                "supervisor_rationale": final_state.get("supervisor_rationale", ""),
                "pdf_report_path": final_state.get("pdf_report_path", None),
            }
        }

    def predict_stream(self, request: Dict[str, Any]) -> Generator[Dict[str, Any], None, None]:
        """Run the multi-agent system and yield progress events."""
        state = self._init_state(request)
        for ev in self.graph.stream(
            state, 
            stream_mode="updates",
            config={"recursion_limit": GRAPH_RECURSION_LIMIT}
        ):
            event_type = ev.get("event")
            node = ev.get("name")
            data = ev.get("data", {}) or {}
            yield {"type": event_type, "node": node, "data": data}

        yield {"type": "final", "node": None, "data": self.predict(request)}


def enable_mlflow(agent: RCAAgent) -> None:
    """Enable MLflow autologging for the provided agent."""
    # Always enable MLflow autologging as per requirement
    deps = get_deps()
    # If mlflow is installed and available via dependencies, configure autolog.
    deps.mlflow.langchain.autolog()


# Public convenience exports -------------------------------------------------
AGENT = RCAAgent()
enable_mlflow(AGENT)

__all__ = [
    "RCAAgent",
    "AGENT",
    "configure_agent_llms",
    "show_llm_configuration",
    "MLFLOW_ENABLED",
    "MAX_OUTER_ITERATIONS",
    "MAX_ANALYZE_PARSE_LOOPS",
    "CONFIDENCE_THRESHOLD",
    "GRAPH_RECURSION_LIMIT",
]


In [0]:
# Example request
req = {
    "user_context": (
        """
        I'm having some issues with Query ID is 01f0a416-cb80-1228-9eda-e3118e89fd48. 
        Task:
        Figure out what went wrong and why it went wrong, see if there is any underlying reason for it.
        """
    ),
    "logs_path": "/Volumes/amruthcatalogtest/default/testsparklogs/00761119/Longer-Bad-00761119_spark/"
}

custom_result = AGENT.predict(req)


In [0]:
print(custom_result)